# Camada Silver: Transformação e Refinamento de Dados

O objetivo deste notebook é processar os dados brutos da camada Bronze, aplicando regras de negócio, limpeza e padronização. Nesta etapa, os dados são transformados em tabelas otimizadas para análise, garantindo a integridade e a qualidade da informação.

**Principais Atividades:**
* **Deduplicação Sênior:** Remoção de registros duplicados em tabelas de dimensões, mantendo apenas a versão mais recente.
* **Padronização de Strings:** Conversão de textos para caixa alta e tradução de status.
* **Cálculos de Engenharia:** Geração de métricas de tempo de entrega e prazos.
* **Tratamento de Nulos:** Preenchimento de valores ausentes em colunas críticas.
* **Consolidação Financeira:** Conversão de valores de BRL para USD utilizando a cotação do dólar ajustada para finais de semana.

In [0]:
# Criar banco de dados Silver
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

from pyspark.sql import Window
from pyspark.sql.functions import col, upper, when, datediff, row_number, last, try_to_timestamp, lit

### Limpeza e Deduplicação de Dimensões
Este bloco define uma função de deduplicação baseada em janelas de tempo, garantindo que Clientes e Vendedores possuam IDs únicos. Os nomes de cidades e estados são padronizados para letras maiúsculas para evitar inconsistências em agrupamentos futuros.

In [0]:
def deduplicar_senior(df, pk_col):
    windowSpec = Window.partitionBy(pk_col).orderBy(col("timestamp_ingestion").desc())
    return df.withColumn("rn", row_number().over(windowSpec)).filter(col("rn") == 1).drop("rn")

# 1. Clientes (dim_consumidores)
df_cust = spark.read.table("bronze.tb_customers").select(
    col("customer_id").alias("id_consumidor"),
    col("customer_zip_code_prefix").alias("prefixo_cep"),
    upper(col("customer_city")).alias("cidade"),
    upper(col("customer_state")).alias("estado"),
    col("timestamp_ingestion")
)
deduplicar_senior(df_cust, "id_consumidor").write.format("delta").mode("overwrite").saveAsTable("silver.dim_consumidores")

# 2. Vendedores (dim_vendedores)
df_sell = spark.read.table("bronze.tb_sellers").select(
    col("seller_id").alias("id_vendedor"),
    col("seller_zip_code_prefix").alias("prefixo_cep"),
    upper(col("seller_city")).alias("cidade"),
    upper(col("seller_state")).alias("estado"),
    col("timestamp_ingestion")
)
deduplicar_senior(df_sell, "id_vendedor").write.format("delta").mode("overwrite").saveAsTable("silver.dim_vendedores")

print("Deduplicação e padronização de Clientes e Vendedores concluída!")

Deduplicação e padronização de Clientes e Vendedores concluída!


### Processamento da Tabela de Pedidos
Nesta etapa, realizamos a tradução dos status dos pedidos de inglês para português. Além disso, são calculadas métricas de performance logística, como o tempo real de entrega em dias e a verificação se o pedido foi entregue dentro do prazo estimado.

In [0]:
df_orders = spark.read.table("bronze.tb_orders")

df_orders_silver = df_orders.select(
    col("order_id").alias("id_pedido"),
    col("customer_id").alias("id_consumidor"),
    # Tradução de Status
    when(col("order_status") == "delivered", "entregue")
    .when(col("order_status") == "canceled", "cancelado")
    .when(col("order_status") == "shipped", "enviado")
    .otherwise("processando").alias("status"),
    col("order_purchase_timestamp").alias("data_compra"),
    col("order_delivered_customer_date").alias("data_entrega_real"),
    col("order_estimated_delivery_date").alias("data_entrega_estimada")
)

# Cálculos de tempo em dias
df_orders_final = df_orders_silver.withColumn(
    "tempo_entrega_dias", datediff(col("data_entrega_real"), col("data_compra"))
).withColumn(
    "tempo_entrega_estimado_dias", datediff(col("data_entrega_estimada"), col("data_compra"))
).withColumn(
    "entrega_no_prazo", when(col("data_entrega_real") <= col("data_entrega_estimada"), "Sim").otherwise("Não")
)

df_orders_final.write.format("delta").mode("overwrite").saveAsTable("silver.fat_pedidos")
print("Tabela silver.fat_pedidos concluída!")

Tabela silver.fat_pedidos concluída!


### Tratamento de Avaliações e Datas
Processamento das avaliações dos consumidores, com foco na limpeza de campos de texto nulos. Utilizamos a função de timestamp para garantir que as datas de criação e resposta estejam em formato adequado para análise temporal, filtrando possíveis inconsistências futuras.

In [0]:
# Ler da Bronze
df_reviews = spark.read.table("bronze.tb_order_reviews")

# Mapeamento e Tratamento
df_reviews_silver = df_reviews.select(
    col("review_id").alias("id_avaliacao"),
    col("order_id").alias("id_pedido"),
    col("review_score").alias("nota_avaliacao"),
    # Preencher nulos conforme o PDF
    when(col("review_comment_title").isNull(), "Sem título").otherwise(col("review_comment_title")).alias("titulo_avaliacao"),
    when(col("review_comment_message").isNull(), "Sem comentário").otherwise(col("review_comment_message")).alias("comentario_avaliacao"),
    # Usar try_to_timestamp para tolerância a falhas
    try_to_timestamp(col("review_creation_date")).alias("data_criacao_avaliacao"),
    try_to_timestamp(col("review_answer_timestamp")).alias("data_resposta_avaliacao")
).filter("data_criacao_avaliacao <= current_timestamp()") # Remover datas no futuro

df_reviews_silver.write.format("delta").mode("overwrite").saveAsTable("silver.fat_avaliacoes_pedidos")
print("Tabela silver.fat_avaliacoes_pedidos concluída!")

Tabela silver.fat_avaliacoes_pedidos concluída!


### Dimensão de Produtos
A tabela de produtos passa pelo processo de deduplicação para garantir que cada SKU seja único. Selecionamos as características físicas dos produtos que serão úteis para análises de frete e logística na camada posterior.

In [0]:
df_prod = spark.read.table("bronze.tb_products")

df_prod_silver = df_prod.select(
    col("product_id").alias("id_produto"),
    col("product_category_name").alias("categoria_produto"),
    col("product_weight_g").alias("peso_gramas"),
    col("product_length_cm").alias("comprimento_cm"),
    col("product_height_cm").alias("altura_cm"),
    col("product_width_cm").alias("largura_cm"),
    col("timestamp_ingestion")
)

# Reutilizando a lógica de pegar o mais recente
windowProd = Window.partitionBy("id_produto").orderBy(col("timestamp_ingestion").desc())
df_prod_final = df_prod_silver.withColumn("rn", row_number().over(windowProd)).filter("rn = 1").drop("rn", "timestamp_ingestion")

df_prod_final.write.format("delta").mode("overwrite").saveAsTable("silver.dim_produtos")
print("Tabela silver.dim_produtos concluída!")

Tabela silver.dim_produtos concluída!


### Consolidação Financeira e Tabela Fato Final
Este bloco realiza o ajuste da cotação do dólar, utilizando a técnica de preenchimento para lacunas (finais de semana). Em seguida, consolidamos os dados de Pedidos e Pagamentos para gerar a tabela fato principal, convertendo os valores financeiros para USD com arredondamento de duas casas decimais. Ao final, aplicamos a otimização física (ZORDER) para melhorar a performance de consulta.

In [0]:
from pyspark.sql.functions import round as spark_round

# 1. Ajustar Dólar (Preencher lacunas de Fim de Semana)
df_dolar = spark.read.table("bronze.tb_cotacao_dolar")
window_dolar = Window.orderBy("dataHoraCotacao").rowsBetween(Window.unboundedPreceding, 0)

df_dolar_ajustado = df_dolar.withColumn(
    "cotacao_venda", last("cotacaoCompra", ignorenulls=True).over(window_dolar)
).select(col("dataHoraCotacao").cast("date").alias("data_cotacao"), "cotacao_venda")

# 2. Juntar tudo na Fato Final (Pedidos + Pagamentos + Dólar)
df_pedidos = spark.read.table("silver.fat_pedidos")
# Agrupando pagamentos por pedido para ter o total por ID
df_pagamentos = spark.read.table("bronze.tb_order_payments").groupBy("order_id").sum("payment_value").withColumnRenamed("sum(payment_value)", "total_brl")

df_final = df_pedidos.join(df_pagamentos, df_pedidos.id_pedido == df_pagamentos.order_id, "left") \
    .join(df_dolar_ajustado, df_pedidos.data_compra.cast("date") == df_dolar_ajustado.data_cotacao, "left")

# 3. Cálculos Finais e Arredondamento (Regra de Negócio: 2 casas decimais) 
df_fat_pedido_total = df_final.select(
    "id_pedido", 
    "id_consumidor", 
    "status",
    spark_round(col("total_brl"), 2).alias("valor_total_pago_brl"),
    spark_round(col("total_brl") / col("cotacao_venda"), 2).alias("valor_total_pago_usd"),
    col("data_compra").alias("data_pedido")
)

# Salvar a tabela final da Silver [cite: 95]
df_fat_pedido_total.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.fat_pedido_total")

# 4. OTIMIZAÇÃO FÍSICA (Exigência: OPTIMIZE com ZORDER) 
spark.sql("OPTIMIZE silver.fat_pedido_total ZORDER BY (id_pedido, data_pedido)")

print("Camada Silver FINALIZADA com sucesso!")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Camada Silver FINALIZADA com sucesso!


### Itens dos Pedidos
Criação da tabela de itens, que serve como ponte entre Pedidos, Produtos e Vendedores. Esta tabela armazena os preços unitários e valores de frete, fundamentais para o cálculo de faturamento e rankings na camada Gold.

In [0]:
# Criando a tabela de itens 
df_itens_bronze = spark.read.table("bronze.tb_order_items")

df_itens_silver = df_itens_bronze.select(
    col("order_id").alias("id_pedido"),
    col("order_item_id").alias("id_item"),
    col("product_id").alias("id_produto"),
    col("seller_id").alias("id_vendedor"),
    col("price").alias("preco_brl"),
    col("freight_value").alias("preco_frete")
)

df_itens_silver.write.format("delta").mode("overwrite").saveAsTable("silver.fat_itens_pedidos")
print("Tabela silver.fat_itens_pedidos criada com sucesso!")

Tabela silver.fat_itens_pedidos criada com sucesso!
